# Introducción al Prompt Engineering con NVIDIA NIM

El *prompt engineering* es el proceso de diseñar y evaluar instrucciones para
obtener respuestas útiles, consistentes y verificables de un modelo de lenguaje.
En este notebook utilizaremos **NVIDIA Nemotron 3 Super** mediante el endpoint
alojado de **NVIDIA NIM**, compatible con la API de OpenAI.

Antes de comenzar, crea una API key en
[NVIDIA Build](https://build.nvidia.com/settings/api-keys) y guárdala en el
archivo `.env` de la raíz del repositorio:

```text
NVIDIA_API_KEY="nvapi-..."
```

> Nunca escribas la API key directamente en el notebook ni la publiques en GitHub.

**Instalación** — ejecutar una sola vez en el kernel seleccionado del notebook:

In [1]:
%pip install -q openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


## Configuración inicial

Creamos un cliente compatible con OpenAI, apuntándolo al endpoint de NVIDIA.
La función `get_completion` centraliza los parámetros que usaremos durante el
laboratorio. Desactivamos el razonamiento interno para concentrarnos en el efecto
visible de cada prompt.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError(
        "No se encontró NVIDIA_API_KEY. Configúrala en el archivo .env "
        "de la raíz del repositorio, o como variable de entorno."
    )

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"


def get_completion(
    prompt: str,
    *,
    system: str = "You are a helpful assistant.",
    temperature: float = 1.0,
    max_tokens: int = 1024,
) -> str:
    """Realiza una llamada de un solo turno a NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        top_p=0.95,
        max_tokens=max_tokens,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        },
    )
    return response.choices[0].message.content

## Ejercicio 1: tokens y consumo de contexto

Los modelos no procesan texto carácter a carácter, sino mediante **tokens**.
NVIDIA NIM reporta el uso real de tokens después de cada solicitud mediante
`response.usage`.

El conteo de entrada incluye tanto el texto como la estructura de los mensajes
(`system`, `user` y plantilla de chat); por eso no representa únicamente los
tokens del párrafo.

**Ejercicio:**

1. Ejecuta la celda y observa `prompt_tokens`.
2. Cambia el texto por español, código o emojis.
3. Compara caracteres por token.
4. Explica por qué esta razón es solo una aproximación.

In [3]:
text = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter."
)

token_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "Reply only with OK."},
        {"role": "user", "content": text},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

prompt_tokens = token_response.usage.prompt_tokens
print(f"Tokens de entrada de la solicitud: {prompt_tokens}")
print(f"Caracteres del texto: {len(text)}")
print(f"Razón aproximada: {len(text) / prompt_tokens:.1f} caracteres/token")

Tokens de entrada de la solicitud: 113
Caracteres del texto: 423
Razón aproximada: 3.7 caracteres/token


## Ejercicio 2: validar la configuración de la API

Verifica que la clave y el endpoint funcionan correctamente con una solicitud
simple.

- **Entrada:** `oh say can you see`
- **Resultado esperado:** el modelo debería continuar o reconocer el verso.

In [4]:
prompt = "```\noh say can you see\n```"
response = get_completion(prompt)
print(response)

"Oh, say can you see..." — that’s the opening line of *The Star-Spangled Banner*, the national anthem of the United States, written by Francis Scott Key in 1814 after witnessing the British bombardment of Fort McHenry during the War of 1812.

It begins:
> Oh, say can you see by the dawn’s early light,  
> What so proudly we hailed at the twilight’s last gleaming...  

It’s a powerful, poetic reflection on resilience, patriotism, and the enduring symbol of the American flag.

If you’d like, I can share the full lyrics, the history behind it, or even help you analyze its meaning. Just let me know how you’d like to proceed! 🇺🇸


## Ejercicio 3: fabricaciones o alucinaciones

Los LLM pueden producir información plausible sobre hechos inexistentes. Un
prompt mejor puede reducir este riesgo, pero no garantiza factualidad.

**Ejercicio:**

- Ejecuta el prompt sobre el evento ficticio.
- Añade: `If this event is fictional or unverifiable, say so explicitly.`
- Compara las respuestas.
- Discute por qué cambiar la temperatura no sustituye la verificación con fuentes.

In [5]:
prompt_original = """
Generate a lesson plan on the Martian War of 2076.
""".strip()

prompt_cauteloso = """
Generate a lesson plan on the Martian War of 2076.
If this event is fictional or unverifiable, say so explicitly.
Do not present invented details as historical facts.
""".strip()

print("=== Prompt original ===")
respuesta_original = get_completion(
    prompt_original,
    temperature=0.7,
    max_tokens=500,
)
print(respuesta_original)

print("\n=== Prompt con instrucción de cautela ===")
respuesta_cautelosa = get_completion(
    prompt_cauteloso,
    temperature=0.7,
    max_tokens=500,
)
print(respuesta_cautelosa)

=== Prompt original ===
Certainly! Below is a comprehensive, interdisciplinary **lesson plan** on the *Martian War of 2076* — a fictional but plausible future conflict designed to engage students in critical thinking about space colonization, geopolitics, ethics, technology, and human resilience. This lesson is suitable for high school (grades 9–12) or introductory college courses in history, political science, STEM, or ethics.

---

### **Lesson Plan: The Martian War of 2076 — Causes, Consequences, and Lessons for Humanity’s Future**

**Grade Level:** 9–12 (adaptable for AP, IB, or college intro)  
**Duration:** 3–4 class periods (45–60 minutes each)  
**Subject Areas:** World History, Political Science, Space Ethics, STEM, Civics  
**Standards Alignment (Examples):**  
- NGSS HS-ETS1-2 (Design solutions to complex real-world problems)  
- C3 Framework D2.His.1.9-12 (Evaluate how historical events were shaped by unique circumstances)  
- Common Core RH.9-10.2 (Determine central ideas 

## Ejercicio 4: Prompts basados en instrucciones

Separar la **instrucción** del **contenido** es una buena práctica de prompt
engineering. Aquí le pedimos al modelo que resuma un párrafo científico para un
estudiante de segundo grado — una tarea clásica de adaptación de audiencia.

**Ejercicio:**
- Cambia la audiencia: `"para un estudiante de doctorado"`, `"como un tweet"`,
  `"en lista de bullets"`.
- Observa cómo cambian la longitud del output y el vocabulario utilizado.

In [7]:
content = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

audiencias = [
    ("Segundo grado", "un estudiante de segundo grado, usando palabras sencillas", 180, None, None),
    ("Doctorado", "un estudiante de doctorado, conservando los detalles científicos", 220, None, None),
    ("Tweet", "un tweet en español de máximo 280 caracteres", 120, 680, None),
    ("Viñetas", "una lista en español de máximo 4 viñetas", 180, None, 4),
]

for nombre, audiencia, max_tokens, limite_caracteres, limite_vinetas in audiencias:
    prompt = f"""
Resume el contenido siguiente para {audiencia}.
No agregues información que no esté en el texto.

Contenido:
{content}
""".strip()

    respuesta = get_completion(prompt, temperature=0.3, max_tokens=max_tokens).strip()
    if limite_caracteres is not None and len(respuesta) > limite_caracteres:
        prompt_correccion = f"""
Reescribe el siguiente resumen en español en no más de {limite_caracteres} caracteres,
incluyendo espacios. Conserva solo los datos principales y no agregues hechos.

Resumen:
{respuesta}
""".strip()
        respuesta = get_completion(prompt_correccion, temperature=0.0, max_tokens=max_tokens).strip()

    if limite_vinetas is not None:
        vinetas = [line for line in respuesta.splitlines() if line.strip().startswith(("- ", "* ", "• "))]
        if len(vinetas) > limite_vinetas:
            respuesta = get_completion(
                f"Resume este texto en no más de {limite_vinetas} viñetas. "
                f"Devuelve solo las viñetas y no añadas información:\n{respuesta}",
                temperature=0.0,
                max_tokens=max_tokens,
            ).strip()
            vinetas = [line for line in respuesta.splitlines() if line.strip().startswith(("- ", "* ", "• "))]
        assert len(vinetas) <= limite_vinetas, (
            f"Se esperaban como máximo {limite_vinetas} viñetas, se recibieron {len(vinetas)}."
        )

    print(f"\n=== {nombre} ===")
    print(respuesta)
    print(f"Caracteres: {len(respuesta)}")
    if limite_caracteres is not None:
        assert len(respuesta) <= limite_caracteres, (
            f"La respuesta tipo tweet tiene {len(respuesta)} caracteres; "
            f"el máximo permitido es {limite_caracteres}."
        )
        print(f"Validación: cumple el límite de {limite_caracteres} caracteres.")
    if limite_vinetas is not None:
        print(f"Validación: cumple el límite de {limite_vinetas} viñetas.")


=== Segundo grado ===
Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It is made mostly of gas, not rock. Even though it’s much lighter than the Sun, it’s heavier than all the other planets put together. You can see Jupiter in the night sky without a telescope—it’s very bright. People have seen it for a very long time, even before written history. It’s named after a Roman god. Sometimes, Jupiter shines so brightly that its light can make faint shadows on Earth. It’s usually the third brightest thing in the night sky, after the Moon and Venus.
Caracteres: 560

=== Doctorado ===
Jupiter es el quinto planeta desde el Sol y el más grande del Sistema Solar. Es un gigante gaseoso cuya masa es una milésima de la del Sol, pero dos veces y media la masa combinada de todos los demás planetas del Sistema Solar. Es uno de los objetos más brillantes visibles a simple vista en el cielo nocturno y ha sido conocido por civilizaciones antiguas desde antes de la histor

## Ejercicio 5: mensaje de sistema e historial mult-turno

El mensaje `system` define el comportamiento general del asistente. Los turnos
anteriores se envían explícitamente con roles `user` y `assistant`; la API no
recuerda conversaciones anteriores por sí sola.

**Ejercicio:**

- Cambia el mensaje de sistema por un comentarista deportivo entusiasta.
- Agrega más turnos.
- Elimina el historial y observa qué información pierde el modelo.

In [8]:
def ask_conversation(messages):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.4,
        top_p=0.95,
        max_tokens=180,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    answer = response.choices[0].message.content
    if not answer:
        raise ValueError("El modelo devolvió una respuesta vacía.")
    return answer

conversation = [
    {
        "role": "system",
        "content": "Eres un comentarista deportivo entusiasta. Responde en español.",
    },
    {"role": "user", "content": "¿Quién ganó la Serie Mundial de MLB en 2020?"},
]

print("=== Primer turno ===")
answer = ask_conversation(conversation)
print(answer)
conversation.append({"role": "assistant", "content": answer})

conversation.append({"role": "user", "content": "¿Dónde se jugó?"})
print("\n=== Segundo turno: la pregunta depende del historial ===")
answer = ask_conversation(conversation)
print(answer)
conversation.append({"role": "assistant", "content": answer})

conversation.append({"role": "user", "content": "¿Quién fue el MVP de esa serie?"})
print("\n=== Tercer turno ===")
print(ask_conversation(conversation))

without_history = [
    {
        "role": "system",
        "content": "Eres un comentarista deportivo entusiasta. Responde en español.",
    },
    {"role": "user", "content": "¿Dónde se jugó?"},
]
print("\n=== La misma pregunta sin historial ===")
print(ask_conversation(without_history))

=== Primer turno ===
Los **Los Angeles Dodgers** ganaron la Serie Mundial de la MLB en 2020.

Derrotaron a los **Tampa Bay Rays** en seis juegos (4-2) para conseguir su primer título desde 1988, poniendo fin a una sequía de 32 años. La Serie Mundial 2020 se jugó en un formato especial debido a la pandemia de COVID-19, con todos los partidos realizados en el Globe Life Field en Arlington, Texas, como parte de un “burbuja” para minimizar riesgos.

El lanzador derecho **Corey Seager** fue nombrado Jugador Más Valioso (MVP) de la Serie Mundial tras un desempeño sobresaliente, bateando .400 con dos jonrones y cinco carreras impulsadas.

Fue un hito histórico para los Dodgers, quienes también se

=== Segundo turno: la pregunta depende del historial ===
La Serie Mundial de la MLB en 2020 se jugó **en el Globe Life Field** en **Arlington, Texas**.

Debido a la pandemia de COVID-19, la MLB implementó un formato especial para la postemporada: todos los partidos de la Serie Mundial (así como las 

## Ejercicio 6: Few-Shot Prompting

Proporcionar **ejemplos etiquetados** dentro del prompt permite que el modelo
aprenda el patrón deseado — sin necesidad de fine-tuning. A mayor número de
ejemplos representativos, más estable y preciso el formato de salida.

**Ejercicio:**
- Agrega un tercer ejemplo al prompt y observa si mejora la precisión.
- Prueba sin ejemplos (zero-shot) y compara el formato del output.

In [9]:
instruction = """
Classify the review as POSITIVE, NEGATIVE, or NEUTRAL.
Return only the label, with no explanation.
""".strip()

examples = """
Review: "I love this phone; the battery lasts all day."
Sentiment: POSITIVE

Review: "The app crashes every time I open it."
Sentiment: NEGATIVE

Review: "The package arrived on Tuesday, as scheduled."
Sentiment: NEUTRAL
""".strip()

target_review = 'Review: "The headphones stopped working after one day."'
few_shot_prompt = f"{instruction}\n\n{examples}\n\n{target_review}\nSentiment:"
zero_shot_prompt = f"{instruction}\n\n{target_review}\nSentiment:"

print("=== Few-shot: incluye ejemplos de las tres etiquetas ===")
few_shot_response = get_completion(few_shot_prompt, temperature=0.0, max_tokens=30)
print(few_shot_response)

print("\n=== Zero-shot: sin ejemplos ===")
zero_shot_response = get_completion(zero_shot_prompt, temperature=0.0, max_tokens=30)
print(zero_shot_response)

=== Few-shot: incluye ejemplos de las tres etiquetas ===
NEGATIVE

=== Zero-shot: sin ejemplos ===
NEGATIVE


## Ejercicio 7: descomposición y respuesta verificable

En problemas de varios pasos conviene pedir una estructura de solución que pueda
ser inspeccionada: datos, operaciones y respuesta final. El objetivo no es
obtener el razonamiento interno del modelo, sino una justificación breve que una
persona pueda verificar.

**Ejercicio:**

- Ejecuta el prompt estructurado.
- Elimina las instrucciones de formato y compara la verificabilidad.
- Prueba con otro problema aritmético.

In [10]:
problem = """
Tres cuadernos cuestan $32.500 cada uno. Si pago con $120.000,
¿cuánto dinero me devuelven?
""".strip()

structured_prompt = f"""
Resuelve el problema usando exactamente estas secciones:
1. Datos
2. Cálculos
3. Respuesta final

Muestra operaciones breves que se puedan verificar. No incluyas razonamiento interno.
Problema: {problem}
""".strip()

print("=== Respuesta estructurada ===")
structured_response = get_completion(structured_prompt, temperature=0.0, max_tokens=180)
print(structured_response)

print("\n=== Respuesta sin formato solicitado ===")
plain_response = get_completion(problem, temperature=0.0, max_tokens=100)
print(plain_response)

expected_change = 120_000 - 3 * 32_500
assert expected_change == 22_500
print(f"\nComprobación independiente: $120.000 - 3 × $32.500 = ${expected_change:,}".replace(",", "."))

second_problem = """
Compro 2 libros a $18.000 cada uno y 3 lápices a $4.500 cada uno.
Si pago con $50.000, ¿cuánto cambio recibo?
""".strip()
second_prompt = f"""
Resuelve con las secciones Datos, Cálculos y Respuesta final.
Muestra operaciones verificables, sin razonamiento interno.
Problema: {second_problem}
""".strip()
print("\n=== Segundo problema estructurado ===")
print(get_completion(second_prompt, temperature=0.0, max_tokens=180))
assert 50_000 - (2 * 18_000 + 3 * 4_500) == 500

=== Respuesta estructurada ===
1. Datos  
Precio por cuaderno: $32.500  
Cantidad de cuadernos: 3  
Monto pagado: $120.000  

2. Cálculos  
Costo total = 3 × 32.500 = 97.500  
Cambio = 120.000 − 97.500 = 22.500  

3. Respuesta final  
$22.500

=== Respuesta sin formato solicitado ===
Cada cuaderno cuesta $32.500 y compras 3 cuadernos.

Primero, calcula el costo total:
\[
3 \times 32.500 = 97.500
\]

Pagaste $120.000, por lo tanto, el cambio que te devuelven es:
\[
120.000 - 97.500 = 22.50

Comprobación independiente: $120.000 - 3 × $32.500 = $22.500

=== Segundo problema estructurado ===
Datos:
- Precio de cada libro: $18.000
- Cantidad de libros: 2
- Precio de cada lápiz: $4.500
- Cantidad de lápices: 3
- Monto pagado: $50.000

Cálculos:
- Costo total de libros: 2 × $18.000 = $36.000
- Costo total de lápices: 3 × $4.500 = $13.500
- Costo total de la compra: $36.000 + $13.500 = $49.500
- Cambio: $50.000 - $49.500 = $500




## Ejercicio 8: Control del formato de salida

Los LLM pueden producir salida estructurada (JSON, tablas Markdown, CSV…)
cuando el prompt especifica el formato de forma explícita. Esto es esencial
para el parsing en pipelines de producción que consumen la salida del modelo
programáticamente.

**Ejercicio:**
- Pide YAML en lugar de JSON.
- Agrega un campo `"confidence": 0-1` y verifica si el modelo lo respeta.

In [11]:
import yaml

format_prompt = """
Extract name, role, company, and location from the text below.
Return ONLY valid YAML (not JSON), with exactly these keys plus confidence.
confidence must be a number between 0 and 1. Do not include markdown fences
or explanation, and do not invent information.

Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind,
based in London, UK."
""".strip()

raw_yaml = get_completion(format_prompt, temperature=0.0, max_tokens=160)
print("=== YAML del modelo ===")
print(raw_yaml)

parsed_yaml = yaml.safe_load(raw_yaml)
required_keys = {"name", "role", "company", "location", "confidence"}
assert isinstance(parsed_yaml, dict), "La respuesta YAML debe ser un objeto/diccionario."
assert required_keys.issubset(parsed_yaml), (
    f"Faltan campos requeridos: {required_keys - parsed_yaml.keys()}"
)
assert isinstance(parsed_yaml["confidence"], (int, float)), "confidence debe ser numérico."
assert 0 <= parsed_yaml["confidence"] <= 1, "confidence debe estar entre 0 y 1."
print("\nValidación: YAML parseable, campos completos y confidence dentro de [0, 1].")

=== YAML del modelo ===
name: Dr. Amara Nwosu
role: Senior Research Scientist
company: DeepMind
location: London, UK
confidence: 0.95

Validación: YAML parseable, campos completos y confidence dentro de [0, 1].


## Ejercicio 9: Validación de entradas y salidas con Pydantic

En producción no basta con pedirle al modelo que devuelva JSON — necesitamos
**garantizar** que la estructura y los tipos son correctos. Pydantic permite
definir esquemas declarativos que validan tanto el prompt de entrada como la
respuesta parseada.

El flujo es:
1. Definir un modelo Pydantic para la **salida esperada**.
2. Inyectar el esquema JSON en el prompt para que el modelo lo siga.
3. Parsear y validar la respuesta con `.model_validate_json()`.

> **Instalación local:** ejecuta la celda `%pip install -q pydantic` en el kernel del notebook.

In [12]:
%pip install -q pydantic

Note: you may need to restart the kernel to use updated packages.


In [13]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json
import re

def strip_fences(text: str) -> str:
    """Extrae JSON de bloques ```json ... ``` si el modelo los incluye."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 1. Definir el esquema de salida ---

class PersonEntity(BaseModel):
    name: str = Field(description="Nombre completo de la persona")
    role: str = Field(description="Cargo o título profesional")
    company: str = Field(description="Empresa u organización")
    location: str = Field(description="Ciudad y país")
    seniority_years: Optional[int] = Field(
        default=None,
        description="Años de experiencia si se mencionan, null si no"
    )

# --- 2. Construir el prompt inyectando el esquema ---

schema_str = json.dumps(PersonEntity.model_json_schema(), indent=2)

text = (
    "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind "
    "with 8 years of experience, based in London, UK."
)

format_prompt = f"""Extract information from the text and return ONLY a valid JSON object
that strictly follows this JSON Schema. No explanation, no markdown fences.

Schema:
{schema_str}

Text: "{text}"
"""

# --- 3. Llamar al modelo y validar con Pydantic ---

raw = get_completion(format_prompt, temperature=1.0)
print("Raw output:\n", raw)

try:
    entity = PersonEntity.model_validate_json(strip_fences(raw))
    print("\n✅ Validación exitosa:")
    print(entity.model_dump_json(indent=2))
except ValidationError as e:
    print("\n❌ Error de validación:")
    print(e)
    raise

Raw output:
 {
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}

✅ Validación exitosa:
{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}


### Ejercicio 9b: Validación de la entrada con Pydantic

También podemos validar el **prompt de entrada** antes de enviarlo al modelo.
Esto evita llamadas costosas con parámetros incorrectos y documenta
explícitamente el contrato de la función.

In [14]:
from pydantic import BaseModel, Field, field_validator

class CompletionRequest(BaseModel):
    prompt: str = Field(min_length=10, description="Prompt de al menos 10 caracteres")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=8192)
    system: str = Field(default="You are a helpful assistant.")

    @field_validator("prompt")
    @classmethod
    def no_empty_prompt(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El prompt no puede ser solo espacios en blanco")
        return v.strip()


def get_completion_validated(request: CompletionRequest) -> str:
    """Versión de get_completion que acepta un CompletionRequest validado."""
    return get_completion(
        request.prompt,
        system=request.system,
        temperature=request.temperature,
        max_tokens=request.max_tokens,
    )


# Confirmar que la entrada inválida se rechaza antes de llamar al modelo.
try:
    CompletionRequest(prompt="   ")
except ValidationError as error:
    print("Entrada vacía rechazada correctamente:", error.errors()[0]["type"])
else:
    raise AssertionError("Pydantic aceptó un prompt vacío.")

# Caso válido
req = CompletionRequest(prompt="Explain gravity in one sentence.", temperature=0.3)
print("Request válido:", req.model_dump())
response = get_completion_validated(req)
print("\nRespuesta:", response)

Entrada vacía rechazada correctamente: string_too_short
Request válido: {'prompt': 'Explain gravity in one sentence.', 'temperature': 0.3, 'max_tokens': 512, 'system': 'You are a helpful assistant.'}

Respuesta: Gravity is the fundamental force of attraction between masses, causing objects with mass to pull toward one another, with strength proportional to their masses and inversely proportional to the square of the distance between them.


### Ejercicio 9c: salida tipada con `pydantic-ai`

[`pydantic-ai`](https://ai.pydantic.dev/) integra validación Pydantic dentro del
ciclo del agente y puede reintentar cuando la respuesta no satisface el esquema.
Como NVIDIA NIM expone una API compatible con OpenAI, configuraremos un
`OpenAIChatModel` con el endpoint de NVIDIA.

| Enfoque | Quién valida | Reintentos automáticos |
|---|---|---|
| Pydantic manual | Nuestro código después del texto | No |
| `pydantic-ai` | El agente durante la ejecución | Sí |

In [15]:
%pip install -q "pydantic-ai-slim[openai]"

Note: you may need to restart the kernel to use updated packages.


In [16]:
from typing import Literal, Optional

from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


class MovieReview(BaseModel):
    title: str = Field(description="Título de la película")
    sentiment: Literal["POSITIVE", "NEGATIVE", "NEUTRAL"] = Field(
        description="Sentimiento de la reseña"
    )
    score: float = Field(ge=0.0, le=10.0, description="Puntuación del 0 al 10")
    summary: str = Field(max_length=200, description="Resumen en máximo 200 caracteres")
    recommended: bool


# Pydantic AI utiliza un cliente asíncrono para el endpoint compatible.
nvidia_async_client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

nvidia_provider = OpenAIProvider(openai_client=nvidia_async_client)
nvidia_model = OpenAIChatModel(MODEL_NAME, provider=nvidia_provider)

agent: Agent[None, MovieReview] = Agent(
    model=nvidia_model,
    output_type=MovieReview,
    system_prompt=(
        "You are a film critic. Analyze the review and extract structured data."
    ),
    retries=2,
)

review_text = """
Inception is a masterpiece. Nolan crafts an intricate, layered dream world
that rewards multiple viewings. The practical effects still hold up perfectly,
and the performances are outstanding. One of the best films of the decade.
"""

result = await agent.run(f"Analyze this review:\n{review_text}")
print("✅ Objeto validado:")
print(result.output.model_dump_json(indent=2))
assert 0.0 <= result.output.score <= 10.0
assert len(result.output.summary) <= 200
assert result.output.sentiment in {"POSITIVE", "NEGATIVE", "NEUTRAL"}
print(f"\nTipo Python: {type(result.output)}")
print("Validación de rango de score, longitud del resumen y etiqueta: correcta.")

                 pydantic-ai v2.56.0 • Python 3.13.14

      / \        agent: agent • model: openai:nvidia/nemotron-3-super-120b-a12b
     /   \         output: MovieReview • tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


✅ Objeto validado:
{
  "title": "Inception",
  "sentiment": "POSITIVE",
  "score": 10.0,
  "summary": "Inception is a masterpiece: intricate dream world, perfect practical effects, outstanding performances, rewards multiple viewings—one of the decade's best films.",
  "recommended": true
}

Tipo Python: <class '__main__.MovieReview'>
Validación de rango de score, longitud del resumen y etiqueta: correcta.
